In [ ]:
import os, shutil
import pandas as pd, numpy as np
# sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
import sys
try:
    import pyLOM
    print('Entorno con pyLOM instalado')
except ImportError as e:
    print(f'Error al importar pylom: {e}')
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder')
    print('Importando desde carpeta local')
from FotR import GANDALF


# VISTAZO A LA PROPUESTA Y COMPARACIÓN CON LO ANTERIOR

In [ ]:
df_inta = pd.read_csv(
    filepath_or_buffer='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/propuestas_INTA/candidates_top100_stage0.csv',
    sep=','
)
df_inta['priority'] = np.linspace(0, len(df_inta)-1, len(df_inta)).astype(int)
#print('Columns in df_INTA:', df_inta.columns.to_list())

df_comp = pd.read_csv(
    filepath_or_buffer='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_#180/metadata/df_post.csv',
    sep = ',',
    index_col=0
)
#print('Columns in df_comp:', df_comp.columns.to_list())

In [ ]:

# Generar imagen de los dos datasets, pintando aoa vs mach con plotly express, coloreando df_inta por prioridad, dejando el otro por la columna dataset (valores string)

import plotly.express as px
import plotly.graph_objects as go

# Scatter de las propuestas INTA
fig_inta = px.scatter(
    df_inta,
    x="mach",
    y="aoa",
    color="priority",
    color_continuous_scale="Viridis",
    hover_data=["re", "priority"],
)

# Scatter de las simulaciones existentes
fig_comp = px.scatter(
    df_comp,
    x="mach",
    y="aoa",
    color="dataset",
    hover_data=["re", "case_idx", "folder"],
)

# Figura final
fig = go.Figure()

# Añadir primero los puntos del dataset
for trace in fig_comp.data:
    trace.update(
        marker=dict(size=8, opacity=0.7, symbol="circle")
    )
    fig.add_trace(trace)


# Añadir los traces de df_inta
for trace in fig_inta.data:
    trace.update(
        name="Propuestas INTA",      # Nombre en la leyenda
        showlegend=True,            # Evita una entrada por cada color
        marker=dict(
            size=8,
            symbol="diamond",
            line=dict(color="black", width=1),
            colorbar=dict(title="Priority"),
            showscale=True,
        ),
    )
    fig.add_trace(trace)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Mach",
    yaxis_title="AoA",

    # Dejar sitio a la derecha
    margin=dict(r=0),

    width=1200,
    height=800,
    legend=dict(
        title="Dataset",
        x=1.18,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
    ),
)
# poner la leyenda encima de la gráfica
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="center",
    x=1
))

# CODA 2025

In [ ]:
try:
    shutil.rmtree("/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_propose_1/")
except:
    print('No la encuentro')
    pass
# import pandas as pd
# df_post = pd.read_csv('/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22/rans3/metadata/df_post.csv', sep=',', usecols=['aoa', 'mach', 'h', 'case_idx'])

In [ ]:

# EAREN = EarendilsLight(FRODO)

# EAREN.help("CODAReader")

dataset_dir = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON'

gdf = GANDALF(
    os.path.join(dataset_dir, "rans3_propose_1"),
    eq_type="rans", 
    num_stages = 2,
    version="flowsimulator2024"
)
# cases = [64, 79, 87, 88, 94]
gdf.define_cases(
    method="external",
    external_dataframe = pd.DataFrame({
        "aoa": df_inta['aoa'],
        "mach": df_inta['mach'],
        # "h": df_inta['h']
    }),
)

# gdf.define_cases(
#     method="Halton",
#     bounds={"AoA": (0.0, 5.0), "Mach": (0.8, 1.1)}, # {"AoA": (0.0, 5.0), "Mach": (0.3, 1.5), 'h': 11000},
#     n_samples=80,
#     peak_ranges=None,
#     seed=42,
# )


In [ ]:
geom_csv = 'eta_0_0.csv'

gdf.define_geom_file(
    geom_file_path=os.path.join(dataset_dir, 'airfoil_files', geom_csv), sep=',', decimal='.', header = 'infer',
    cols_idx=[1, 3],
    normalize = False
)

c = gdf.array_ptos[:, 0].max() - gdf.array_ptos[:, 0].min()
cm = -c/4 # la malla lo tiene ya ajustado a x=0 el borde de ataque


In [ ]:
print(gdf.df_cases.columns)
T, P, rho = GANDALF.Backpack.isa_atmosphere(h=11000)
mu = GANDALF.Backpack.Sutherland_law(mu0 = 1.716e-5, T=T, Treference=255.55)

gdf.compute_param(
    name = 'Re',
    formula = "L * mach*sqrt(1.4*287*T)*rho/mu",
    externals={
        "rho": rho,
        "mu": mu,
        "L": c,
        "T": T
    }
)
print(gdf.df_cases.columns)

In [ ]:
gdf.generate_folders(
    base_files = ["run_sst_v4.py", "run.sh"],
    mesh_path=os.path.join(dataset_dir, "sources", "mesh_f22_0_0_v7.msh"),
    script_dir=os.path.join(dataset_dir, "sources"),
    folder_fmt = "aoa_{aoa:.4f}_mach_{mach:.4f}",
    overwrite=False,
    update_base_files=True,
    data_to_update={
        "AOA_PLACEHOLDER": "aoa",
        "MACH_PLACEHOLDER": "mach",
        # "ALT_PLACEHOLDER": "h",
        "PYTHON_FILE_PLACEHOLDER": "run_sst_v4.py",
        "CHORD_PLACEHOLDER": str(c),
        "CM_PLACEHOLDER": str(cm)
    }
)

In [ ]:
# df_sacar = gdf.df_cases.copy()
# df_sacar['AoA'] = pd.to_numeric(df_sacar['AoA'])
# df_sacar['Mach'] = pd.to_numeric(df_sacar['Mach'])
# df_definitivo = df_sacar.round({'AoA': 4, 'Mach': 4})
# df_definitivo['Coef_Area'] = c
# df_definitivo.to_csv(os.path.join(gdf.root_dir,'metadata', 'df_cases.csv'), index=False)
gdf.df_cases['Coef_Area'] = c
gdf.df_cases.to_csv(os.path.join(gdf.root_dir,'metadata', 'df_cases.csv'), index=False)

In [ ]:
gdf.assign_jobs(
    file_sh="run.sh",
    nodes=[f'n00{n}' for n in [1, 3, 4, 5, 6, 7]],
    cpus_per_job=48,
    submit=True,
    )

In [ ]:
gdf.submit_cases()

# CAMBIAR TRABAJOS DE NODO X AL NODO Y

In [ ]:
dataset_dir = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON'

gdf = GANDALF(
    os.path.join(dataset_dir, "rans3_propose_0"),
    eq_type="rans", 
    num_stages = 2,
    version="flowsimulator2024"
)

gdf.recover_pending_jobs(
    broken_node = 'n003',
    replacement_node = 'n005',
    file_sh = 'run.sh',
    dry_run = True
)